In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')
from sklearn.model_selection import train_test_split, cross_val_score,StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
ConfusionMatrixDisplay, roc_auc_score,average_precision_score, precision_recall_curve)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
df = pd.read_csv(r"C:\Users\niluc\Downloads\PROJECT\M-pesa\Data\mpesa_clean.csv")
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

print(f'features: {X.shape}')
print(f'fraud rate: {y.mean():.4f}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train fraud: {y_train.sum()} | Test fraud {y_test.sum()}') 

#pipeline
rf_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=42, k_neighbors=5)),
    ('model',  RandomForestClassifier(n_estimators=200, max_depth=12,
                   class_weight='balanced', random_state=42, n_jobs=-1))
])
xgb_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=42)),
    ('model',  XGBClassifier(n_estimators=200,
                   learning_rate=0.1, max_depth=6, random_state=42, subsample=0.8,
                            colsample_bytree=0.8,eval_metrics='logloss', scale_pos_weight=13.3,n_jobs=-1))
])
lr_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=42)),
    ('model',  LogisticRegression(max_iter=1000, class_weight='balanced'))
])

lr_no_smote_pipe=ImbPipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42))
])

pipelines ={
    'Logistic Regression': lr_pipe,
    'Random Forest': rf_pipe,
    'XGBoost': xgb_pipe,
    'LR No SMOTE':lr_no_smote_pipe
}
results = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, prob)
    ap = average_precision_score(y_test, prob)
    results[name] = {'auc' :auc, 'ap' :ap, 'prob' :prob, 'pipe' :pipe, 'pred':pred}
    print(f'\n=== {name}===')
    print(classification_report(y_test,pred,target_names=['Legit','Fraud']))
    print(f'ROC_AUC:{auc:.4f} | Avg Precision: {ap:.4f}')

    best_name = max(results, key=lambda k: results[k]['ap'])
    best = results[best_name]
    print(f'\nBest: {best_name} AUC={best["auc"]:.4f} AP={best["ap"]:.4f}')

    plt.figure(figsize=(8,5))
for name, res in results.items():
    p,r,_ =precision_recall_curve(y_test,res['prob'])
    plt.plot(r, p, label=f'{name} (AP={res["ap"]:.3f})')
    plt.xlabel('Recall');plt.ylabel('Precision')
    plt.title('Precision-Recall -M_PESA Fraud Detection')
    plt.legend();
    plt.tight_layout()
    plt.savefig('mpesa_pr_curve.png',dpi=120)

#confusion matrix
cm = confusion_matrix(y_test, best['pipe'].predict(X_test))
ConfusionMatrixDisplay(cm,display_labels=['Legit','Fraud']).plot(cmap='Reds')
plt.title(f'Confusion Matrix-{best_name}')
plt.tight_layout();
plt.savefig('mpesa_confusion_matrix.png', dpi=120)

#feature importance
xgb_model = results['XGBoost']['pipe'].named_steps['model']
fi=pd.Series(rf_model.feature_importances_, index=X.columns)
fi.nlargest(10).sort_values().plot(kind='barh',figsize=(8,5), color='#1B8CA6')
plt.title('Top 10 Fraud indicators')
plt.tight_layout();
plt.savefig('mpesa_feature_importances.png', dpi=120)
print('n\Top 5 fraud indicators:')
print(fi.nlargest(5).to_string())

from sklearn.metrics import recall_score, precision_score
print('\nThreshold Analysis (KES Fraud Recovery)')
for t in [0.3,0.4,0.5,0.6]:
    p_t =(best['prob'] >=t).astype(int)
    rec = recall_score(y_test,p_t)
    prec = precision_score(y_test,p_t)
    fraud_caught = int(rec* y_test.sum())
print(f'Threshold={t}: Recall={rec:.3f} Precision={prec:.3f} Fraud caught={fraud_caught}')

best_pipe =best['pipe']
joblib.dump(best_pipe, 'models/mpesa_pipeline.pkl')
joblib.dump(list(X.columns), 'models/mpesa_feature_names.pkl')
print('Pipeline saved.')



features: (500000, 15)
fraud rate: 0.0319
Train fraud: 12751 | Test fraud 3188

=== Logistic Regression===
              precision    recall  f1-score   support

       Legit       0.97      0.67      0.79     96812
       Fraud       0.04      0.47      0.08      3188

    accuracy                           0.66    100000
   macro avg       0.51      0.57      0.44    100000
weighted avg       0.94      0.66      0.77    100000

ROC_AUC:0.5987 | Avg Precision: 0.0473

Best: Logistic Regression AUC=0.5987 AP=0.0473

=== Random Forest===
              precision    recall  f1-score   support

       Legit       0.97      0.87      0.92     96812
       Fraud       0.05      0.22      0.08      3188

    accuracy                           0.85    100000
   macro avg       0.51      0.54      0.50    100000
weighted avg       0.94      0.85      0.89    100000

ROC_AUC:0.5633 | Avg Precision: 0.0450

Best: Logistic Regression AUC=0.5987 AP=0.0473


C:\Users\niluc\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [16:26:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "eval_metrics" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



=== XGBoost===
              precision    recall  f1-score   support

       Legit       0.99      1.00      0.99     96812
       Fraud       0.90      0.74      0.81      3188

    accuracy                           0.99    100000
   macro avg       0.94      0.87      0.90    100000
weighted avg       0.99      0.99      0.99    100000

ROC_AUC:0.9387 | Avg Precision: 0.8226

Best: XGBoost AUC=0.9387 AP=0.8226

=== LR No SMOTE===
              precision    recall  f1-score   support

       Legit       0.97      0.66      0.79     96812
       Fraud       0.04      0.47      0.08      3188

    accuracy                           0.66    100000
   macro avg       0.51      0.57      0.44    100000
weighted avg       0.94      0.66      0.77    100000

ROC_AUC:0.6038 | Avg Precision: 0.0476

Best: XGBoost AUC=0.9387 AP=0.8226
n\Top 5 fraud indicators:
day_of_week           0.242334
hour                  0.235821
sender_txn_count      0.159232
sender_county_risk    0.076517
channel_en